@author: eveomett

# Lab 3: MAUP and data.  See details on Canvas page

## Make sure to say where/when you got your data!

In [1]:
import sys
!{sys.executable} -m pip install --only-binary=:all: "maup>=1.1.0" numpy pandas geopandas gerrychain
import pandas as pd
import geopandas as gpd
import maup
from maup import smart_repair
import time
import os

maup.progress.enabled = True

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# paths
population_path = ".\ma_pl2020_b\ma_pl2020_p1_b.shp"
election_path = ".\ma_2024_gen_prec\ma_2024_gen_all_prec\ma_2024_gen_all_prec.shp"
county_path = ".\ma_pl2020_cnty\ma_pl2020_cnty.shp"

# load data
pop_df = gpd.read_file(population_path)
county_df = gpd.read_file(county_path)
elec_df = gpd.read_file(election_path)

print("population rows:", len(pop_df))
print("county rows:", len(county_df))
print("election rows:", len(elec_df))

population rows: 107278
county rows: 14
election rows: 2383


In [4]:
# move everything to one UTM CRS
utm_crs = elec_df.estimate_utm_crs()

pop_df = pop_df.to_crs(utm_crs)
county_df = county_df.to_crs(utm_crs)
elec_df = elec_df.to_crs(utm_crs)

print(pop_df.crs)
print(county_df.crs)
print(elec_df.crs)

# doctor checks before repair
print("population doctor:", maup.doctor(pop_df))
print("county doctor:", maup.doctor(county_df))
print("election doctor:", maup.doctor(elec_df))

EPSG:32619
EPSG:32619
EPSG:32619


100%|█████████████████████████████| 107278/107278 [03:42<00:00, 482.95it/s]


population doctor: True


100%|██████████████████████████████████████| 14/14 [00:00<00:00, 47.69it/s]


county doctor: True


100%|█████████████████████████████████| 2383/2383 [00:07<00:00, 299.32it/s]


There are 519 overlaps.
There are 77 holes.
There are some invalid geometries.
election doctor: False


In [5]:
# repair precincts inside counties
snap_precision = 8  # higher = more precise snapping to county boundaries
final_df = smart_repair(elec_df, nest_within_regions=county_df, snap_precision=snap_precision)

# convert small rook boundaries to queen adjacencies
min_rook_length = 30  # borders shorter than this (meters) become point contacts
final_df = smart_repair(final_df, min_rook_length=min_rook_length)

print("final doctor:", maup.doctor(final_df))


100%|██████████████████████████████████████| 14/14 [00:00<00:00, 85.82it/s]


Snapping all geometries to a grid with precision 10^( -3 ) to avoid GEOS errors.


100%|██████████████████████████████████████| 14/14 [00:00<00:00, 15.17it/s]


Identifying overlaps...


100%|█████████████████████████████████| 8753/8753 [00:09<00:00, 889.53it/s]


...

Resolving overlaps...
Filling gaps...


Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]


Converting small rook adjacencies to queen...


100%|█████████████████████████████████| 2383/2383 [00:07<00:00, 304.34it/s]


final doctor: True


In [6]:
# assign 2020 blocks to precincts and add total population
precinct_assignments = maup.assign(pop_df.geometry, final_df.geometry)

final_df["TOTPOP"] = pop_df["P0010001"].groupby(precinct_assignments).sum()
final_df["TOTPOP"] = final_df["TOTPOP"].fillna(0)

# population check: block total must equal precinct total
block_pop = pop_df["P0010001"].sum()
precinct_pop = final_df["TOTPOP"].sum()
assert block_pop == precinct_pop, f"Population mismatch: blocks={block_pop}, precincts={precinct_pop}"
print(f"Population check passed: {block_pop:,} == {precinct_pop:,}")

# keep district data from the precinct file
final_df["CD"] = final_df["CONG_DIST"]

# save shapefile
if not os.path.exists("MA"):
    os.mkdir("MA")
final_df.to_file(r"MA\MA.shp")

print("STACK SUMMARY")
print("precinct file used:", election_path)
print("population file used:", population_path)
print("county file used:", county_path)
print("doctor after repair:", maup.doctor(final_df))
print("total population in blocks:", block_pop)
print("total population in final precincts:", precinct_pop)
print("number of districts:", final_df["CD"].nunique())
print(final_df["CD"].dropna().unique())


100%|██████████████████████████████████| 2383/2383 [00:32<00:00, 73.31it/s]


Population check passed: 7,029,917 == 7,029,917
STACK SUMMARY
precinct file used: .\ma_2024_gen_prec\ma_2024_gen_all_prec\ma_2024_gen_all_prec.shp
population file used: .\ma_pl2020_b\ma_pl2020_p1_b.shp
county file used: .\ma_pl2020_cnty\ma_pl2020_cnty.shp


100%|█████████████████████████████████| 2383/2383 [00:07<00:00, 327.52it/s]


doctor after repair: True
total population in blocks: 7029917
total population in final precincts: 7029917
number of districts: 9
['08' '03' '09' '01' '06' '02' '05' '04' '07']


In [7]:
# Run this first to see what election columns are available in final_df
print(final_df.columns.tolist())

['UNIQUE_ID', 'COUNTYFP', 'City/Town', 'Ward', 'Pct', 'County', 'CONG_DIST', 'SLDL_DIST', 'SLDU_DIST', 'G24PREDHAR', 'G24PRELOLI', 'G24PREOOTH', 'G24PRERTRU', 'G24PREUAYY', 'G24PREUCRU', 'G24PREUSON', 'G24PREUSTE', 'G24PREUWES', 'G24USSDWAR', 'G24USSOOTH', 'G24USSRDEA', 'GCON01DNEA', 'GCON01OOTH', 'GCON01UMIL', 'GCON02DMCG', 'GCON02OOTH', 'GCON02USHE', 'GCON03DTRA', 'GCON03OOTH', 'GCON04DAUC', 'GCON04OOTH', 'GCON04UFAD', 'GCON05DCLA', 'GCON05OOTH', 'GCON06DMOU', 'GCON06OOTH', 'GCON07DPRE', 'GCON07OOTH', 'GCON08DLYN', 'GCON08OOTH', 'GCON08RBUR', 'GCON09DKEA', 'GCON09OOTH', 'GCON09RSUL', 'GSLB01DLOU', 'GSLB01OOTH', 'GSLB01RCHA', 'GSLB02DHAW', 'GSLB02OOTH', 'GSLB02UBEL', 'GSLB02WNEL', 'GSLB03DDOH', 'GSLB03OOTH', 'GSLB04OOTH', 'GSLB04RHOW', 'GSLB05DHAD', 'GSLB05OOTH', 'GSLB05RTHU', 'GSLB06DFIO', 'GSLB06OOTH', 'GSLB07DSIL', 'GSLB07OOTH', 'GSLB08DOUE', 'GSLB08OOTH', 'GSLB08RTHR', 'GSLB08UGEL', 'GSLB08UHAD', 'GSLB08USOA', 'GSLB09DMAR', 'GSLB09OOTH', 'GSLB10DSYL', 'GSLB10OOTH', 'GSLB10RPIR', '

In [8]:
DEM_COL_1 = "G24PREDHAR"   # 2024 President - Harris (Democrat)
REP_COL_1 = "G24PRERTRU"   # 2024 President - Trump (Republican)
DEM_COL_2 = "G24USSDWAR"   # 2024 US Senate - Democrat
REP_COL_2 = "G24USSRDEA"   # 2024 US Senate - Republican

# save column names so analysis.ipynb can load them
import json as _json
with open("config.json", "w") as _f:
    _json.dump({"DEM_COL_1": DEM_COL_1, "REP_COL_1": REP_COL_1,
                "DEM_COL_2": DEM_COL_2, "REP_COL_2": REP_COL_2}, _f)
print("config.json saved")


config.json saved
